# Name embedding analysis

Check whether the 16x uint8 `name` column in `product_properties` retains semantic structure after quantisation.

In [1]:
import re
from collections import Counter
from pathlib import Path

import numpy as np
import polars as pl
from scipy.spatial.distance import cdist
from sklearn.metrics import mutual_info_score

DATA_DIR = Path.cwd().parent.parent / 'DATA'
PROPS_PATH = DATA_DIR / 'product_properties.parquet'
SEED = 1337
N_SAMPLE = 50_000
N_CATEGORIES_SAMPLE = 200
print(PROPS_PATH, PROPS_PATH.exists())

/home/mofu/code/thesis/GITREPO/beau-min-maai-portfolio/code/DATA/product_properties.parquet True


In [2]:
def parse_name_bytes(s):
    return [int(x) for x in re.findall(r'\d+', s)][:16]

pp = pl.read_parquet(PROPS_PATH)
print(f'{pp.height:,} items')

name_raw = [parse_name_bytes(s) for s in pp['name'].to_list()]
name_arr = np.array(
    [nb[:16] + [0] * max(0, 16 - len(nb)) for nb in name_raw],
    dtype=np.float32,
)
categories = pp['category'].to_numpy().astype(np.int64)
print(name_arr.shape, categories.shape)

1,534,050 items


(1534050, 16) (1534050,)


## Test 1: byte value distribution per position

In [3]:
for pos in range(16):
    vals = name_arr[:, pos].astype(int)
    unique = len(np.unique(vals))
    counts = np.array(list(Counter(vals).values()), dtype=np.float64)
    probs = counts / counts.sum()
    entropy = -np.sum(probs * np.log2(probs))
    print(f'  Byte {pos:2d}: {unique:3d} unique | entropy={entropy:.3f}/8.000 ({entropy/8*100:.1f}%)')

all_vals = name_arr.flatten().astype(int)
counts = np.array(list(Counter(all_vals).values()), dtype=np.float64)
probs = counts / counts.sum()
overall_entropy = -np.sum(probs * np.log2(probs))
print(f'\n  Overall: {len(np.unique(all_vals))} unique | entropy={overall_entropy:.3f}/8.000 ({overall_entropy/8*100:.1f}%)')

  Byte  0: 256 unique | entropy=7.973/8.000 (99.7%)


  Byte  1: 256 unique | entropy=7.963/8.000 (99.5%)


  Byte  2: 256 unique | entropy=7.972/8.000 (99.6%)


  Byte  3: 256 unique | entropy=7.972/8.000 (99.6%)


  Byte  4: 256 unique | entropy=7.968/8.000 (99.6%)


...

  Byte 12: 256 unique | entropy=7.971/8.000 (99.6%)


  Byte 13: 256 unique | entropy=7.967/8.000 (99.6%)


  Byte 14: 256 unique | entropy=7.979/8.000 (99.7%)


  Byte 15: 256 unique | entropy=7.965/8.000 (99.6%)



  Overall: 256 unique | entropy=7.990/8.000 (99.9%)


## Test 2: intra- vs inter-category cosine similarity

In [4]:
rng = np.random.default_rng(SEED)
cat_ids, cat_counts = np.unique(categories, return_counts=True)
valid_cats = cat_ids[cat_counts >= 10]
sampled_cats = rng.choice(valid_cats, size=min(N_CATEGORIES_SAMPLE, len(valid_cats)), replace=False)

intra_sims, inter_sims = [], []
for cat in sampled_cats:
    mask = categories == cat
    cat_vecs = name_arr[mask]
    if len(cat_vecs) < 2:
        continue
    n_pairs = min(50, len(cat_vecs))
    idx = rng.choice(len(cat_vecs), size=n_pairs, replace=False)
    vecs = cat_vecs[idx]
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    vecs_n = vecs / norms
    sim = vecs_n @ vecs_n.T
    triu = np.triu_indices(len(vecs_n), k=1)
    if len(triu[0]) > 0:
        intra_sims.extend(sim[triu].tolist())
    other_idx = rng.choice(np.where(~mask)[0], size=n_pairs, replace=False)
    other_vecs = name_arr[other_idx]
    other_norms = np.linalg.norm(other_vecs, axis=1, keepdims=True)
    other_norms = np.where(other_norms == 0, 1, other_norms)
    other_n = other_vecs / other_norms
    inter_sims.extend((vecs_n @ other_n.T).flatten().tolist())

intra_mean, inter_mean = np.mean(intra_sims), np.mean(inter_sims)
intra_std, inter_std = np.std(intra_sims), np.std(inter_sims)
print(f'  Intra: {intra_mean:.4f} +- {intra_std:.4f}  (n={len(intra_sims):,})')
print(f'  Inter: {inter_mean:.4f} +- {inter_std:.4f}  (n={len(inter_sims):,})')
print(f'  Diff:  {intra_mean - inter_mean:.4f}')

  Intra: 0.7840 +- 0.0977  (n=184,233)
  Inter: 0.7580 +- 0.0926  (n=376,743)
  Diff:  0.0260


## Test 3: nearest-neighbour category agreement

In [5]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(name_arr), size=min(N_SAMPLE, len(name_arr)), replace=False)
vecs = name_arr[idx]
cats = categories[idx]

batch_size, n = 5000, len(vecs)
same_cat = 0
for start in range(0, n, batch_size):
    end = min(start + batch_size, n)
    dists = cdist(vecs[start:end], vecs, metric='euclidean')
    for i in range(end - start):
        dists[i, start + i] = np.inf
    nn_idx = np.argmin(dists, axis=1)
    same_cat += np.sum(cats[start:end] == cats[nn_idx])

nn_rate = same_cat / n
n_cats = len(np.unique(cats))
uniform = 1.0 / n_cats
cat_counts = Counter(cats)
weighted = sum((c / n) ** 2 for c in cat_counts.values())
print(f'  NN same-cat:      {nn_rate:.4f} ({nn_rate*100:.2f}%)')
print(f'  Uniform chance:   {uniform:.4f} ({uniform*100:.2f}%)')
print(f'  Weighted chance:  {weighted:.4f} ({weighted*100:.2f}%)')
print(f'  Lift vs weighted: {nn_rate / weighted:.2f}x')

  NN same-cat:      0.1421 (14.21%)
  Uniform chance:   0.0002 (0.02%)
  Weighted chance:  0.0025 (0.25%)
  Lift vs weighted: 57.17x


## Test 4: mutual information (byte -> category)

In [6]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(name_arr), size=min(N_SAMPLE, len(name_arr)), replace=False)
cats = categories[idx]
mis = []
for pos in range(16):
    byte_vals = name_arr[idx, pos].astype(int)
    mi = mutual_info_score(byte_vals, cats)
    mis.append(mi)
    print(f'  Byte {pos:2d}: MI = {mi:.4f} nats')
random_bytes = rng.integers(0, 256, size=len(cats))
random_mi = mutual_info_score(random_bytes, cats)
print(f'\n  Random baseline MI: {random_mi:.4f} nats')
print(f'  Mean name byte MI:  {np.mean(mis):.4f} nats')

  Byte  0: MI = 3.4491 nats
  Byte  1: MI = 3.4194 nats
  Byte  2: MI = 3.2181 nats
  Byte  3: MI = 3.3720 nats
  Byte  4: MI = 3.3467 nats
  Byte  5: MI = 3.3161 nats
  Byte  6: MI = 3.3717 nats
  Byte  7: MI = 3.4104 nats
  Byte  8: MI = 3.3997 nats
  Byte  9: MI = 3.4050 nats
  Byte 10: MI = 3.3758 nats
  Byte 11: MI = 3.3475 nats
  Byte 12: MI = 3.3510 nats
  Byte 13: MI = 3.3513 nats
  Byte 14: MI = 3.3842 nats
  Byte 15: MI = 3.3932 nats

  Random baseline MI: 2.2004 nats
  Mean name byte MI:  3.3695 nats


## Test 5: PCA variance explained

In [7]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(name_arr), size=min(N_SAMPLE, len(name_arr)), replace=False)
vecs = name_arr[idx]
vecs_c = vecs - vecs.mean(axis=0)
cov = np.cov(vecs_c, rowvar=False)
eig = np.linalg.eigvalsh(cov)[::-1]
explained = eig / eig.sum()
cumulative = np.cumsum(explained)
for i in range(16):
    print(f'  PC {i+1:2d}: {explained[i]*100:5.2f}%  (cum {cumulative[i]*100:5.1f}%)')
print(f'\n  Uniform expectation: ~{100/16:.2f}% per PC')
print(f'  Top-3 PCs: {cumulative[2]*100:.1f}% (vs {3*100/16:.1f}% if random)')

  PC  1: 13.46%  (cum  13.5%)
  PC  2:  6.31%  (cum  19.8%)
  PC  3:  6.20%  (cum  26.0%)
  PC  4:  6.17%  (cum  32.1%)
  PC  5:  6.08%  (cum  38.2%)
  PC  6:  5.95%  (cum  44.2%)
  PC  7:  5.85%  (cum  50.0%)
  PC  8:  5.81%  (cum  55.8%)
  PC  9:  5.74%  (cum  61.6%)
  PC 10:  5.64%  (cum  67.2%)
  PC 11:  5.60%  (cum  72.8%)
  PC 12:  5.54%  (cum  78.4%)
  PC 13:  5.51%  (cum  83.9%)
  PC 14:  5.44%  (cum  89.3%)
  PC 15:  5.37%  (cum  94.7%)
  PC 16:  5.33%  (cum 100.0%)

  Uniform expectation: ~6.25% per PC
  Top-3 PCs: 26.0% (vs 18.8% if random)
